# Skin Lesion Classification — Data Exploration

Exploratory analysis of the **HAM10000 / ISIC 2018 Task 3** dataset.

1. Dataset statistics and class distribution
2. Sample images per class
3. Augmentation strategy — one representative example per class
4. Split statistics

In [ ]:
!pip install albumentations>=1.3.0,"<2.0.0" -q

In [ ]:
import sys, os
from google.colab import drive
drive.mount('/content/drive')

# Clone repo and add src to path
if not os.path.exists('/content/skin-lesion-classifier-CNN'):
    os.system('git clone https://github.com/daorre1202/skin-lesion-classifier-CNN.git /content/skin-lesion-classifier-CNN')
sys.path.insert(0, '/content/skin-lesion-classifier-CNN/src')
print('✓ Path configurado')

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from collections import Counter
from pathlib import Path
from PIL import Image

from skin_classifier.data.splits import merge_groundtruth_csvs, split_dataset
from skin_classifier.data.transforms import build_transforms
from skin_classifier.utils.io import index_images

# ── Paths — adjust if your Drive folder has a different name ──────────
CSV_DIR    = '/content/drive/MyDrive/ISIC2018_Task3/Groundtruth'
SOURCE_DIR = '/content/drive/MyDrive/ISIC2018_Task3/ISIC2018'

CLASSES = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
CLASS_NAMES = {
    'MEL':   'Melanoma',
    'NV':    'Melanocytic Nevus',
    'BCC':   'Basal Cell Carcinoma',
    'AKIEC': 'Actinic Keratoses',
    'BKL':   'Benign Keratosis',
    'DF':    'Dermatofibroma',
    'VASC':  'Vascular Lesion',
}
MALIGNANT = {'MEL', 'BCC', 'AKIEC'}
COLOR_MAL = '#c0392b'
COLOR_BEN = '#2980b9'
COLORS    = [COLOR_MAL if c in MALIGNANT else COLOR_BEN for c in CLASSES]

random.seed(42)
print('✓ Imports listos')

## 1. Cargar etiquetas y estadísticas básicas

In [ ]:
label_map = merge_groundtruth_csvs(CSV_DIR)
id2path   = index_images(SOURCE_DIR)
id2path   = {k: v for k, v in id2path.items() if k in label_map}

class_counts = Counter(label_map[iid] for iid in id2path)
total = sum(class_counts.values())

print(f'Total imágenes: {total}')
print(f'\n{"Clase":8s} {"Nombre completo":42s} {"N":>6s}  {"  %":>6s}')
print('-' * 68)
for cls in CLASSES:
    n   = class_counts.get(cls, 0)
    pct = n / total * 100
    tag = '  ← maligna' if cls in MALIGNANT else ''
    print(f'{cls:8s} {CLASS_NAMES[cls]:42s} {n:6d}  ({pct:5.1f}%){tag}')

## 2. Distribución de clases

In [ ]:
counts  = [class_counts.get(c, 0) for c in CLASSES]
pcts    = [n / total * 100 for n in counts]
labels  = [f'{c}\n{CLASS_NAMES[c]}' for c in CLASSES]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('HAM10000 — Class Distribution', fontsize=14, fontweight='bold')

# ── Left: bar chart ───────────────────────────────────────────────────
bars = axes[0].bar(range(len(CLASSES)), counts, color=COLORS,
                   edgecolor='white', linewidth=0.5)
for bar, n, pct in zip(bars, counts, pcts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{n}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=8)
axes[0].set_xticks(range(len(CLASSES)))
axes[0].set_xticklabels(
    [c for c in CLASSES], fontsize=10, fontweight='bold')
axes[0].set_xlabel(
    'MEL=Melanoma  NV=Melanocytic Nevus  BCC=Basal Cell Ca.  '
    'AKIEC=Actinic Ker.  BKL=Benign Ker.  DF=Dermatofibroma  VASC=Vascular',
    fontsize=7.5, color='gray')
axes[0].set_ylabel('Number of images')
axes[0].set_title('Absolute counts')
axes[0].grid(axis='y', alpha=0.3)
legend_patches = [
    mpatches.Patch(color=COLOR_MAL, label='Malignant / pre-malignant'),
    mpatches.Patch(color=COLOR_BEN, label='Benign'),
]
axes[0].legend(handles=legend_patches, fontsize=9)

# ── Right: horizontal bar — avoids label overlap entirely ────────────
y_pos = range(len(CLASSES))
hbars = axes[1].barh(list(y_pos), pcts, color=COLORS,
                     edgecolor='white', linewidth=0.5)
for i, (pct, n) in enumerate(zip(pcts, counts)):
    axes[1].text(pct + 0.3, i, f'{pct:.1f}%  (n={n})',
                 va='center', fontsize=9)
axes[1].set_yticks(list(y_pos))
axes[1].set_yticklabels(
    [f'{c} — {CLASS_NAMES[c]}' for c in CLASSES], fontsize=9
)
axes[1].set_xlabel('Percentage of dataset (%)')
axes[1].set_title('Relative proportions')
axes[1].set_xlim(0, max(pcts) * 1.25)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'NV domina el dataset con {pcts[CLASSES.index("NV")]:.1f}% — desbalanceo severo.')
print(f'Ratio de desbalanceo NV/DF: {counts[CLASSES.index("NV")]/counts[CLASSES.index("DF")]:.1f}×')

## 3. Imágenes representativas por clase

In [ ]:
N_COLS = 4  # imágenes por clase

# Recoger N_COLS imágenes por clase con orden fijo
samples = {cls: [] for cls in CLASSES}
for iid, path in id2path.items():
    cls = label_map[iid]
    if len(samples[cls]) < N_COLS:
        samples[cls].append(path)

fig = plt.figure(figsize=(N_COLS * 2.8, len(CLASSES) * 2.8))
fig.suptitle('Sample images — 4 per class', fontsize=13,
             fontweight='bold', y=1.01)

for row, cls in enumerate(CLASSES):
    color  = COLOR_MAL if cls in MALIGNANT else COLOR_BEN
    tag    = ' ⚠ MALIGNANT' if cls in MALIGNANT else ''

    for col in range(N_COLS):
        ax = fig.add_subplot(len(CLASSES), N_COLS, row * N_COLS + col + 1)
        if col < len(samples[cls]):
            img = np.array(
                Image.open(samples[cls][col]).convert('RGB').resize((160, 160))
            )
            ax.imshow(img)
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(2.5)
        ax.set_xticks([]); ax.set_yticks([])

        # Etiqueta de clase solo en la primera columna
        if col == 0:
            ax.set_ylabel(
                f'{cls}\n{CLASS_NAMES[cls]}{tag}',
                fontsize=8.5, color=color, fontweight='bold',
                rotation=0, labelpad=90, va='center'
            )

fig.text(0.01, -0.005,
         'Red border = malignant / pre-malignant  |  '
         'Blue border = benign',
         fontsize=9, color='gray', style='italic')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Augmentation — ejemplo real por clase

Cada clase recibe el nivel de augmentation asignado automáticamente por su ratio de desbalanceo.
Se muestra la imagen original y 3 versiones augmentadas con ese nivel.

In [ ]:
from skin_classifier.data.dataset import (
    AUG_THR_LIGHT, AUG_THR_MEDIUM, AUG_THR_HEAVY
)

transforms = build_transforms(224)
max_count  = max(class_counts.values())

# Calcular nivel de augmentation de cada clase
def get_aug_level(cls):
    r = max_count / class_counts[cls]
    if r < AUG_THR_LIGHT:   return 0
    if r < AUG_THR_MEDIUM:  return 1
    if r < AUG_THR_HEAVY:   return 2
    return 3

LEVEL_NAMES = {
    0: 'Nivel 0 — sin augmentation',
    1: 'Nivel 1 — giros y rotaciones',
    2: 'Nivel 2 — + transformaciones afines',
    3: 'Nivel 3 — + ruido y desenfoque',
}

N_AUG = 3   # versiones augmentadas a mostrar
MEAN  = np.array([0.485, 0.456, 0.406])
STD   = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(
    len(CLASSES), N_AUG + 1,
    figsize=((N_AUG + 1) * 2.6, len(CLASSES) * 2.8)
)
fig.suptitle(
    'Augmentation strategy — one example per class\n'
    'Each class receives the level assigned by its imbalance ratio',
    fontsize=12, fontweight='bold'
)

for row, cls in enumerate(CLASSES):
    level   = get_aug_level(cls)
    ratio   = max_count / class_counts[cls]
    color   = COLOR_MAL if cls in MALIGNANT else COLOR_BEN
    sample  = samples[cls][0]  # primera imagen de esta clase
    img_np  = np.array(Image.open(sample).convert('RGB'))

    for col in range(N_AUG + 1):
        ax = axes[row, col]
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(1.5)

        if col == 0:
            # Original
            ax.imshow(img_np)
            ax.set_title('Original', fontsize=8)
            ax.set_ylabel(
                f'{cls} — {CLASS_NAMES[cls]}\n'
                f'ratio={ratio:.1f}×  {LEVEL_NAMES[level]}',
                fontsize=7.5, color=color, fontweight='bold',
                rotation=0, labelpad=130, va='center'
            )
        else:
            # Augmented
            aug_t = transforms['aug'][level](image=img_np.copy())['image']
            img_d = np.clip(aug_t.numpy().transpose(1,2,0) * STD + MEAN, 0, 1)
            ax.imshow(img_d)
            ax.set_title(f'Aug {col}', fontsize=8)

plt.tight_layout()
plt.savefig('augmentation_per_class.png', dpi=120, bbox_inches='tight')
plt.show()
print('Augmentation levels por clase:')
for cls in CLASSES:
    level = get_aug_level(cls)
    ratio = max_count / class_counts[cls]
    boost = ' ★ CLINICAL BOOST → nivel 2' if cls == 'MEL' and level < 2 else ''
    print(f'  {cls:6s}  ratio={ratio:5.1f}×  →  {LEVEL_NAMES[level]}{boost}')

## 5. Estadísticas del split 60/20/20

In [ ]:
prep_train, prep_val, prep_test = split_dataset(
    label_map, id2path, train_ratio=0.6, val_ratio=0.2, seed=42
)

rows = []
for cls in CLASSES:
    tr = len(prep_train.get(cls, []))
    va = len(prep_val.get(cls,   []))
    te = len(prep_test.get(cls,  []))
    rows.append({
        'Class': cls, 'Full name': CLASS_NAMES[cls],
        'Train': tr, 'Val': va, 'Test': te, 'Total': tr+va+te,
        'Type': 'Malignant' if cls in MALIGNANT else 'Benign',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nTotals — Train: {df['Train'].sum()} | "
      f"Val: {df['Val'].sum()} | Test: {df['Test'].sum()}")
print('\nSplit estratificado: cada clase mantiene proporciones ~60/20/20.')
print('Determinista: misma semilla → mismo split siempre.')

## Resumen

Observaciones clave de la exploración:

- **Desbalanceo severo**: NV representa ~66% de las imágenes. Las clases más raras (DF, VASC) tienen menos de 200 imágenes — ratios de desbalanceo de hasta 48×.
- **Prioridad clínica**: MEL, BCC y AKIEC son malignas o premalignas y requieren alta sensibilidad. Su baja representación refuerza la necesidad de WeightedRandomSampler y calibración de umbrales clínicos.
- **Augmentation graduada**: cada clase recibe el nivel de augmentation correspondiente a su ratio. MEL recibe un boost clínico adicional al nivel 2 independientemente de su ratio calculado.
- **Split reproducible**: el split estratificado 60/20/20 con IDs ordenados léxicamente garantiza que el mismo código produce siempre la misma partición, sin data leakage entre sesiones.